# Очистка данных: Deals

In [40]:
import os
import re
import datetime
import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import help_130625_dam as h

pd.set_option('display.float_format', '{:.2f}'.format)
np.set_printoptions(suppress=True, precision=2)

# Пути к данным
RAW_DATA_DIR = os.path.join('..', 'Sources')
CLEANED_DIR  = os.path.join('..', 'data', 'cleaned')

DEALS_INPUT = os.path.join(RAW_DATA_DIR, 'Deals (Done).xlsx')
DEALS_OUTPUT = os.path.join(CLEANED_DIR, 'deals_clean.pkl')
MAPPING_INPUT = os.path.join(CLEANED_DIR, 'contact_mapping.pkl')

In [41]:
def time_to_seconds(t):
    """Конвертирует datetime.time → float (секунды) или np.nan."""
    if isinstance(t, datetime.time):
        return float(t.hour * 3600 + t.minute * 60 + t.second)
    try:
        # Пытаемся сконвертировать, если это уже число или строка
        val = float(t)
        return val if np.isfinite(val) else np.nan
    except (ValueError, TypeError):
        return np.nan
    


## Загрузка и первичный осмотр

In [42]:
# Contact Name и Id читаем как str, чтобы избежать
# потери точности при промежуточном float64 (проявляется при наличии NaN в колонке)
df = pd.read_excel(DEALS_INPUT, dtype={'Contact Name': str, 'Id': str})

# Переименование столбцов в snake_case
df.columns = [h.to_snake(c) for c in df.columns]

# Переименование contact_name в contact_id для единообразия с другими таблицами
df = df.rename(columns={'contact_name': 'contact_id'})

# Используем df_de_raw, чтобы не накапливать ошибки при повторных запусках 
# для нормализации уровня немецкого языка
df_de_raw = df['level_of_deutsch']
n_before = len(df)

print(f'Форма: {df.shape}')
h.descr_df(df, include='all', show_sample_rows=True)

Форма: (21595, 23)


,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,id,str,21593,2,21593,5805028000056864695,5805028000056859489,5805028000056832357,NaN,NaN,NaN,NaN
1,deal_owner_name,str,21564,31,27,Ben Hall,Ulysses Adams,Ulysses Adams,NaN,NaN,NaN,NaN
2,closing_date,str,14645,6950,359,NaN,NaN,21.06.2024,NaN,NaN,NaN,NaN
3,quality,str,19340,2255,6,NaN,NaN,D - Non Target,NaN,NaN,NaN,NaN
4,stage,str,21593,2,13,New Lead,New Lead,Lost,NaN,NaN,NaN,NaN
5,lost_reason,str,16124,5471,21,NaN,NaN,Non target,NaN,NaN,NaN,NaN
6,page,str,21593,2,34,/eng/test,/at-eng,/at-eng,NaN,NaN,NaN,NaN
7,campaign,str,16067,5528,154,03.07.23women,NaN,engwien_AT,NaN,NaN,NaN,NaN
8,sla,object,15533,6062,13357,NaN,NaN,00:26:43,NaN,NaN,NaN,NaN
9,content,str,14147,7448,187,v16,NaN,b1-at,NaN,NaN,NaN,NaN


In [43]:
# Проверка уникальности технического ID
ids_counts = df['id'].nunique()
print(f'Уникальных ID сделок: {ids_counts} ({"все ID уникальны" if ids_counts == n_before else "есть дубликаты по ID!"})')

# Поиск бизнес-дубликатов по ключевым полям: клиент, время создания, продукт, сумма оплаты
# Убираем offer_total_amount из ключей, так как он может варьироваться или быть пустым
BUSINESS_KEYS = ['contact_id', 'created_time', 'product', 'initial_amount_paid']

# Проверяем наличие колонок перед поиском дублей
search_cols = [c for c in BUSINESS_KEYS if c in df.columns]
biz_dupes = df.duplicated(subset=search_cols).sum()
lost_dupes = df[df['lost_reason'].astype(str).str.contains('дубликат|duplicate', case=False, na=False)]

print(f'Обнаружено бизнес-дубликатов по ключам {search_cols}: {biz_dupes}')
print(f"Найдено сделок с lost_reason='дубликат': {len(lost_dupes)}")

if biz_dupes > 0:
    print("\nПример бизнес-дубликатов (ключи совпадают, ID разные):")
    display(df[df.duplicated(subset=search_cols, keep=False)].sort_values(search_cols).head(4))

Уникальных ID сделок: 21593 (есть дубликаты по ID!)
Обнаружено бизнес-дубликатов по ключам ['contact_id', 'created_time', 'product', 'initial_amount_paid']: 24
Найдено сделок с lost_reason='дубликат': 1771

Пример бизнес-дубликатов (ключи совпадают, ID разные):


,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_id,city,level_of_deutsch
21588,5805028000000970006,Jane Smith,04.07.2023,E - Non Qualified,Lost,Duplicate,eng/digital-marketing,03.07.23women,NaN,b3,...,NaN,NaN,04.07.2023 07:10,NaN,NaN,NaN,NaN,5805028000000979006,NaN,NaN
21589,5805028000000948010,Jane Smith,29.08.2023,B - Medium,Lost,needs time to think,eng/digital-marketing,03.07.23women,NaN,b3,...,NaN,NaN,04.07.2023 07:10,NaN,NaN,NaN,NaN,5805028000000979006,NaN,NaN
21548,5805028000001369097,Bob Brown,08.07.2023,E - Non Qualified,Lost,Duplicate,eng/digital-marketing,performancemax_digitalmarkt_ru_DE,NaN,_{region_name}_,...,NaN,NaN,08.07.2023 11:39,NaN,NaN,0,0,5805028000001347003,NaN,NaN
21549,5805028000001355009,Bob Brown,08.07.2023,E - Non Qualified,Lost,Duplicate,eng/digital-marketing,performancemax_digitalmarkt_ru_DE,NaN,_{region_name}_,...,NaN,NaN,08.07.2023 11:39,NaN,NaN,0,0,5805028000001347003,NaN,NaN


In [44]:
# Удаляем технический шум (полностью пустые строки без Id)
df = df.dropna(subset=['id']).reset_index(drop=True)
print(f'Строк после удаления пустых Id: {len(df)}')

Строк после удаления пустых Id: 21593


## Дедупликация

In [45]:
# Удаление записей с явной пометкой 'дубликат' в CRM (Пометил менеджер)
before_crm = len(df)
df = df[~df['lost_reason'].astype(str).str.contains('дубликат|duplicate', case=False, na=False)]
print(f'Удалено записей с lost_reason="дубликат": {before_crm - len(df)}')

# Удаление полных дубликатов (если вдруг остались)
before_full = len(df)
df = df.drop_duplicates()
print(f'Удалено полных дубликатов: {before_full - len(df)}')

# Удаление бизнес-дубликатов (ключи совпадают, ID разные)
# Состав ключей: клиент, время создания, продукт, сумма оплаты
before_biz = len(df)
df = df.drop_duplicates(subset=['contact_id', 'created_time', 'product', 'initial_amount_paid'], keep='last')
print(f'Удалено найденных нами бизнес-дубликатов: {before_biz - len(df)}')

print(f'\nИтого строк после дедупликации: {len(df)}')

Удалено записей с lost_reason="дубликат": 1771
Удалено полных дубликатов: 0
Удалено найденных нами бизнес-дубликатов: 7

Итого строк после дедупликации: 19815


## 6. Id и Contact Id: object → Int64 (безопасно)

In [46]:
# id и contact_id читаем как str. конвертируем в Int64 напрямую через Python int(),
# минуя float64-промежуток, который округляет 19-значные числа.
# Заодно убираем '.0' / '.00', которые Excel может добавить при сохранении числа как float.

df['id'] = pd.array([h.str_to_int64(v) for v in df['id']], dtype='Int64')
df['contact_id'] = pd.array([h.str_to_int64(v) for v in df['contact_id']], dtype='Int64')

print('id:        ', df['id'].dtype, '| NaN:', df['id'].isna().sum())
print('contact_id:', df['contact_id'].dtype, '| NaN:', df['contact_id'].isna().sum())

# Подготовка: Группировка сделок БЕЗ contact_id под одним ID -1
df.loc[df['contact_id'].isna(), 'contact_id'] = -1

id:         Int64 | NaN: 0
contact_id: Int64 | NaN: 47


## Типы данных: даты

In [47]:
# Created Time: '21.06.2024 15:30'  → datetime
df['created_time'] = pd.to_datetime(df['created_time'], dayfirst=True, errors='coerce')

# Closing Date: '21.06.2024' → datetime (NaT = сделка ещё открыта)
df['closing_date'] = pd.to_datetime(df['closing_date'], dayfirst=True, errors='coerce')

print('created_time:', df['created_time'].dtype, '| NaT:', df['created_time'].isna().sum())
print('closing_date:', df['closing_date'].dtype, '| NaT:', df['closing_date'].isna().sum())
print(f'Диапазон created_time: {df["created_time"].min()}  →  {df["created_time"].max()}')

created_time: datetime64[us] | NaT: 0
closing_date: datetime64[us] | NaT: 6680
Диапазон created_time: 2023-07-03 17:03:00  →  2024-06-21 15:30:00


## SLA: время ответа → секунды

> `SLA` хранит объекты `datetime.time` (hh:mm:ss). Для анализа удобнее хранить как **целое число секунд**.

In [48]:
print('sla — тип и примеры значений до обработки:')
print(df['sla'].dtype)

# Принудительная конвертация всего столбца в float
df['sla'] = df['sla'].apply(time_to_seconds)

# Фильтрация экстремальных выбросов (> 24 часов)
outliers_mask = df['sla'] > 24 * 3600
if outliers_mask.any():
    print(f"Обнаружено {outliers_mask.sum()} аномально высоких значений SLA (>48ч), сбрасываем их.")
    df.loc[outliers_mask, 'sla'] = np.nan

# Восполнение медианой по менеджеру (transform('median') работает корректно на уникальных строках)
if 'deal_owner_name' in df.columns:
    manager_medians = df.groupby('deal_owner_name', observed=True)['sla'].transform('median')
    global_median = df['sla'].median()
    
    n_nan_before = df['sla'].isna().sum()
    df['sla'] = df['sla'].fillna(manager_medians).fillna(global_median)
    n_filled = n_nan_before - df['sla'].isna().sum()
    print(f'Восполнено пропусков SLA: {n_filled}')

# Финальное приведение к целочисленному типу с поддержкой NULL
df['sla'] = df['sla'].round(0).astype('Int32')

print(f'\nsla после обработки — тип: {df["sla"].dtype}')
print(f'Осталось NaN в SLA: {df["sla"].isna().sum()}')
if df["sla"].notna().any():
    print(f'Диапазон: {df["sla"].min()} сек  →  {df["sla"].max()} сек')
    print(f'Среднее: {df["sla"].mean():.0f} сек ({df["sla"].mean()/60:.1f} мин)')
    print(f'Медиана: {df["sla"].median():.0f} сек ({df["sla"].median()/60:.1f} мин)')
else:
    print("Данные SLA отсутствуют.")

sla — тип и примеры значений до обработки:
object


Восполнено пропусков SLA: 6578

sla после обработки — тип: Int32
Осталось NaN в SLA: 0
Диапазон: 3 сек  →  86292 сек
Среднее: 21766 сек (362.8 мин)
Медиана: 14154 сек (235.9 мин)


## 5. Числовые поля: очистка сумм

In [49]:
AMOUNT_COLS = ['initial_amount_paid', 'offer_total_amount']
for col in AMOUNT_COLS:
    df[col] = clean_amount(df[col])
    print(f'{col}: {df[col].dtype}, min={df[col].min()}, max={df[col].max():,.0f}, NaN={df[col].isna().sum()}')

initial_amount_paid: float64, min=0.0, max=11,500, NaN=0
offer_total_amount: float64, min=0.0, max=11,500, NaN=0


## 7. Level of Deutsch: нормализация

> Поле содержит 215 уникальных значений: смесь кириллицы и латиницы (например `а2` vs `A2`, `б1` vs `B1`),
> а также свободный текст (адреса, фразы). Приводим к стандарту CEFR (A0–C2), остальное → `Unknown`.

In [50]:
# Список всех значений, которые не соответствуют паттерну [A1-C2] после очистки кириллицы
CYR_TO_LAT = str.maketrans('абвсАБВС', 'abvcABVC')

# ТАБЛИЦА ЗАМЕН: 
LEVEL_MAP = {
    'а': 'a', 'А': 'A',
    'б': 'b', 'Б': 'B',
    'в': 'b', 'В': 'B',
    'с': 'c', 'С': 'C'
}

def normalize_deutsch(value):
    if pd.isna(value):
        return pd.NA
    
    orig_s = str(value).strip()
    s_lower = orig_s.lower()
    
    # 1. Специальные случаи для A0
    a0_exact = ['0', 'no', 'none', '?', '-', 'нет', 'a'] # A без цифры -> A0
    a0_keywords = ['никакой', 'нулевой', 'не учил', 'не учила', 'anfanger', 'beginner', 'начальный']
    if s_lower in a0_exact or any(keyword in s_lower for keyword in a0_keywords):
        return 'A0'
        
    # 2. Специальные случаи для других уровней
    if s_lower == 'в': return 'B1'
    if s_lower == 'f2': return 'A2'
    if s_lower == 'c': return 'C1'
    
    # 3. ПОИСК УРОВНЕЙ: Любая буква [AaBbCcАаБбВвСс] + цифра [012]
    match = re.search(r'([AaBbCcАаБбВвСс][0-2])', orig_s)
    if match:
        found = match.group(1)
        letter = found[0]
        digit = found[1]
        letter_lat = LEVEL_MAP.get(letter, letter).upper()
        return f"{letter_lat}{digit}"
    
    # 4. Маппинг остальных ключевых слов
    words_map = {
        'intermediate': 'B1', 'средний': 'B1',
        'advanced': 'C1'
    }
    for word, level in words_map.items():
        if word in s_lower:
            return level
            
    return 'Unclear'

# Применяем очистку
# Используем df_raw, чтобы не накапливать ошибки при повторных запусках
# df_raw = pd.read_excel(DEALS_INPUT)
df['level_of_deutsch'] = df_de_raw.apply(normalize_deutsch)

# Анализ оставшихся нестандартных значений (для проверки)
def get_non_standard(value):
    if pd.isna(value) or value == 'Unclear': return value
    # Если значение уже в стандарте A1-C2, возвращаем None
    if re.search(r'\b([AaBbCc][012])\b', str(value)):
        return None
    return value

non_std_count = (df['level_of_deutsch'] == 'Unclear').sum()
print(f"Всего строк с нераспознанным уровнем (Unclear): {non_std_count}")

print("\n--- ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ УРОВНЕЙ ---")
display(df['level_of_deutsch'].value_counts().to_frame())

Всего строк с нераспознанным уровнем (Unclear): 15

--- ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ УРОВНЕЙ ---


,count
level_of_deutsch,
B1,816
B2,169
A2,150
A0,40
C1,28
A1,25
Unclear,15
C2,3


## 6. Группировка стадий (Funnel)

Для построения воронки продаж и анализа конверсии мы объединяем 13 детальных стадий CRM в 4 укрупненные бизнес-группы:
1. **Marketing/Lead** — новые регистрации и потенциальные интересы.
2. **Active Sales** — стадия переговоров, консультаций и пробных периодов.
3. **Won/Paid** — успешное завершение сделки (оплата).
4. **Lost** — закрытые сделки без оплаты.

In [51]:
# ГРУППИРОВКА СТАДИЙ
# Цель: упростить воронку до 4 бизнес-этапов

STAGE_GROUPS = {
    'New Lead': 'Marketing/Lead',
    'Registered on Webinar': 'Marketing/Lead',
    'Registered on Offline Day': 'Marketing/Lead',
    'Need To Call': 'Active Sales',
    'Need to Call - Sales': 'Active Sales',
    'Need a consultation': 'Active Sales',
    'Qualificated': 'Active Sales',
    'Test Sent': 'Active Sales',
    'Call Delayed': 'Active Sales',
    'Waiting For Payment': 'Active Sales',
    'Free Education': 'Active Sales',
    'Payment Done': 'Won/Paid',
    'Lost': 'Lost'
}

# Создание новой колонки
df['stage_group'] = df['stage'].map(STAGE_GROUPS).fillna('Other')

# 2. Проверка результатов
print("Распределение стадий по группам:")
display(df['stage_group'].value_counts().to_frame())

Распределение стадий по группам:


,count
stage_group,
Lost,13996
Active Sales,2808
Marketing/Lead,2155
Won/Paid,856


In [52]:
# Проверка пропусков в датасете
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
display(
    pd.DataFrame({'Пропуски': missing, '% пропусков': missing_pct})
    .query('Пропуски > 0')
    .sort_values('Пропуски', ascending=False)
)
print('Строк без пропусков:', df.dropna().shape[0])

,Пропуски,% пропусков
payment_type,19332,97.56
months_of_study,18977,95.77
level_of_deutsch,18569,93.71
city,17305,87.33
education_type,16559,83.57
course_duration,16282,82.17
product,16278,82.15
term,7716,38.94
closing_date,6680,33.71
content,6021,30.39


Строк без пропусков: 3


## 7. Восполнение пропусков (Backfill) на основе Contact Name

> Если для одного и того же клиента (`Contact Name`) в разных сделках заполнены разные поля (Source, City и т.д.), 
> мы можем «протянуть» эти значения на пустые строки этого же клиента.

In [53]:
# Поля для заполнения
COLS_TO_FILL = ['source', 'campaign', 'city', 'level_of_deutsch', 'deal_owner_name']
COLS_CHECK = COLS_TO_FILL + ['course_duration', 'offer_total_amount']

df['is_buyer'] = (df['initial_amount_paid'] > 0) & (df['months_of_study'] > 0)

# Приводим колонки к object перед восполнением
for col in COLS_TO_FILL:
    if col in df.columns:
        df[col] = df[col].astype(object)

# Считаем "пропуски" до восполнения. 
# Для финансовых полей считаем 0 как пропуск, так как мы их будем восполнять.
missing_before = df[COLS_TO_FILL].isnull().sum()
missing_before['course_duration'] = df['course_duration'].isna().sum()
missing_before['offer_total_amount'] = (df['offer_total_amount'].isna() | (df['offer_total_amount'] == 0)).sum()

# Backfill по contact_id (протягиваем данные между разными сделками одного клиента)
df = df.sort_values(['contact_id', 'created_time'])
for col in COLS_TO_FILL:
    if col in df.columns:
        df[col] = df.groupby('contact_id', group_keys=False)[col].apply(lambda x: x.ffill().bfill())

# Восполнение deal_owner_name из контактов
CONTACTS_CLEAN = os.path.join('..', 'data', 'cleaned', 'contacts_clean.pkl')
if os.path.exists(CONTACTS_CLEAN):
    contacts = pd.read_pickle(CONTACTS_CLEAN)
    contacts['id'] = contacts['id'].astype('Int64')
    contact_owner_map = contacts.drop_duplicates('id').set_index('id')['contact_owner_name']
    
    mask_isna = df['deal_owner_name'].isna()
    df.loc[mask_isna, 'deal_owner_name'] = df.loc[mask_isna, 'contact_id'].map(contact_owner_map)

# Восполнение course_duration и offer_total_amount по продукту (Mode / Median)
if 'product' in df.columns:
    # Ограничиваем выборку только записями, где продукт не 'Unknown'
    valid_prod_mask = (df['product'].notna()) & (df['product'] != 'Unknown')
    
    # Длительность курса (Мода)
    prod_duration_map = (
        df[valid_prod_mask]
        .groupby('product', observed=True)['course_duration']
        .apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )
    
    # Полная стоимость оффера (Медиана)
    # Берем медиану только из тех строк, где сумма > 0
    prod_offer_map = (
        df[valid_prod_mask & (df['offer_total_amount'] > 0)]
        .groupby('product', observed=True)['offer_total_amount']
        .median()
    )

    # Применяем восполнение только для NaN
    mask_dur = df['course_duration'].isna()
    df.loc[mask_dur, 'course_duration'] = df.loc[mask_dur, 'product'].map(prod_duration_map)
    
    mask_offer = (df['offer_total_amount'].isna()) | (df['offer_total_amount'] == 0)
    df.loc[mask_offer, 'offer_total_amount'] = df.loc[mask_offer, 'product'].map(prod_offer_map)

# Считаем итоги
missing_after = df[COLS_TO_FILL].isnull().sum()
missing_after['course_duration'] = df['course_duration'].isna().sum()
missing_after['offer_total_amount'] = (df['offer_total_amount'].isna() | (df['offer_total_amount'] == 0)).sum()

filled = missing_before - missing_after

print('\nДинамика восполнения пропусков (0 в офферах теперь считаются как пропуски):')
display(pd.DataFrame({
    'Было (NaN или 0)': missing_before,
    'Восполнено': filled,
    'Осталось пропусков': missing_after
}))

lost_no_date = (df['stage'] == 'Lost') & (df['closing_date'].isna())
df.loc[lost_no_date, 'closing_date'] = df.loc[lost_no_date, 'created_time']

print(f"Заполнено '0' для месяцев обучения в {len(df[df['stage'] == 'Lost'])} потерянных сделках.")
print(f"Заполнена дата закрытия (равна дате создания) для {lost_no_date.sum()} потерянных сделок.")


Динамика восполнения пропусков (0 в офферах теперь считаются как пропуски):


,Было (NaN или 0),Восполнено,Осталось пропусков
source,0,0,0
campaign,4234,857,3377
city,17305,654,16651
level_of_deutsch,18569,376,18193
deal_owner_name,29,29,0
course_duration,16282,0,16282
offer_total_amount,16526,257,16269


Заполнено '0' для месяцев обучения в 13996 потерянных сделках.
Заполнена дата закрытия (равна дате создания) для 1640 потерянных сделок.


In [54]:
# Поля, где пропуск = отсутствие информации → заполняем 'Unknown'
FILL_UNKNOWN = ['quality', 'lost_reason', 'campaign', 'content', 'term',
                'payment_type', 'product', 'education_type', 'city', 'level_of_deutsch', 'deal_owner_name']

for col in FILL_UNKNOWN:
    if col in df.columns:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            if isinstance(df[col].dtype, pd.CategoricalDtype):
                if 'Unknown' not in df[col].cat.categories:
                    df[col] = df[col].cat.add_categories('Unknown')
            else:
                df[col] = df[col].astype(object)
            
            df[col] = df[col].fillna('Unknown')
            print(f'{col}: заполнено {n_miss} пропусков → "Unknown"')

# Числовые поля (длительность, сумма оффера, месяцы обучения) → заполняем 0

FILL_ZERO = ['course_duration', 'months_of_study', 'offer_total_amount']

for col in FILL_ZERO:
    if col in df.columns:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            df[col] = df[col].fillna(0.0)
            print(f'{col}: заполнено {n_miss} пропусков → 0.0')

print("\nФинальный статус пропусков в ключевых колонках:")
remaining = df.isnull().sum()
display(
    pd.DataFrame({'Пропуски': remaining, '%': (remaining / len(df) * 100).round(2)})
    .query('Пропуски > 0')
    .sort_values('Пропуски', ascending=False)
)

quality: заполнено 2230 пропусков → "Unknown"
lost_reason: заполнено 5463 пропусков → "Unknown"
campaign: заполнено 3377 пропусков → "Unknown"
content: заполнено 6021 пропусков → "Unknown"
term: заполнено 7716 пропусков → "Unknown"
payment_type: заполнено 19332 пропусков → "Unknown"
product: заполнено 16278 пропусков → "Unknown"
education_type: заполнено 16559 пропусков → "Unknown"
city: заполнено 16651 пропусков → "Unknown"
level_of_deutsch: заполнено 18193 пропусков → "Unknown"
course_duration: заполнено 16282 пропусков → 0.0
months_of_study: заполнено 18977 пропусков → 0.0
offer_total_amount: заполнено 16269 пропусков → 0.0

Финальный статус пропусков в ключевых колонках:


,Пропуски,%
closing_date,5040,25.44


In [55]:
# ПРЕОБРАЗОВАНИЕ ТИПОВ
# Оптимизируем типы данных для уменьшения потребления памяти и удобства анализа

# 1. Категориальные поля
CAT_COLS = [
    'stage', 'stage_group', 'deal_owner_name', 'product', 
    'quality', 'source', 'campaign', 'city', 'level_of_deutsch',
    'payment_type', 'page', 'lost_reason', 'term', 'content'
]

for col in CAT_COLS:
    if col in df.columns:
        df[col] = df[col].astype('category')

# 2. Поля с ограниченным набором значений
if 'education_type' in df.columns:
    df['education_type'] = df['education_type'].astype('category')

# 3. Числовые типы
# course_duration и months_of_study могут содержать NaN, поэтому используем Int32
INT_COLS = ['course_duration', 'months_of_study']
for col in INT_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int32')

In [56]:
h.descr_df(df, include='all', show_sample_rows=True)
print(f"Объем памяти: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,id,Int64,19815,0,19815,5805028000004028088,5805028000005183749,5805028000005695023,5805028000000921600.00,5805028000030290944.00,5805028000030272512.00,5805028000056893440.00
1,deal_owner_name,category,19815,0,27,Charlie Davis,Jane Smith,Jane Smith,<NA>,<NA>,<NA>,<NA>
2,closing_date,datetime64[us],14775,5040,1981,2023-09-20 00:00:00,2023-08-23 00:00:00,2023-09-10 00:00:00,<NA>,<NA>,<NA>,<NA>
3,quality,category,19815,0,6,C - Low,E - Non Qualified,E - Non Qualified,<NA>,<NA>,<NA>,<NA>
4,stage,category,19815,0,13,Lost,Lost,Lost,<NA>,<NA>,<NA>,<NA>
5,lost_reason,category,19815,0,21,Changed Decision,Doesn't Answer,Stopped Answering,<NA>,<NA>,<NA>,<NA>
6,page,category,19815,0,33,/,/,/,<NA>,<NA>,<NA>,<NA>
7,campaign,category,19815,0,154,nina,nina,nina,<NA>,<NA>,<NA>,<NA>
8,sla,Int32,19815,0,11199,3746,4,6090,3.00,21765.94,14154.00,86292.00
9,content,category,19815,0,185,Unknown,Unknown,Unknown,<NA>,<NA>,<NA>,<NA>


Объем памяти: 1.85 MB


## 11. Итоговый осмотр

## 12. Сохранение

In [57]:
# Применяем маппинг дублей контактов (сформирован в 01_cleaning_contacts)
if os.path.exists(MAPPING_INPUT):
    contact_mapping = pd.read_pickle(MAPPING_INPUT)
    affected = df['contact_id'].isin(contact_mapping.keys()).sum()
    df['contact_id'] = df['contact_id'].replace(contact_mapping.to_dict())
    print(f"Маппинг контактов применён: {affected} сделок перепривязаны к мастер-контактам.")
else:
    print("Файл contact_mapping.pkl не найден. Сначала выполните 01_cleaning_contacts.")

os.makedirs(os.path.dirname(DEALS_OUTPUT), exist_ok=True)
df.to_pickle(DEALS_OUTPUT)
df.to_excel(DEALS_OUTPUT.replace('.pkl', '.xlsx'))

summary_data = {
    'Метрика': [
        'Строк исходно', 'Строк после очистки', 'Удалено дубликатов',
        'Уникальных сделок (id)', 'Диапазон created_time',
        'stage (уникальных)', 'Медиана SLA, мин', 'Пропуски после заполнения'
    ],
    'Значение': [
        n_before, len(df), n_before - len(df),
        df['id'].nunique(),
        f'{df["created_time"].min().date()} → {df["created_time"].max().date()}',
        df['stage'].nunique(),
        round(df['sla'].median(), 1),
        df.isnull().sum().sum()
    ]
}

print(f'Сохранено: {DEALS_OUTPUT}')
display(pd.DataFrame(summary_data))

Маппинг контактов применён: 26 сделок перепривязаны к мастер-контактам.
Сохранено: ..\data\cleaned\deals_clean.pkl


,Метрика,Значение
0,Строк исходно,21595
1,Строк после очистки,19815
2,Удалено дубликатов,1780
3,Уникальных сделок (id),19815
4,Диапазон created_time,2023-07-03 → 2024-06-21
5,stage (уникальных),13
6,"Медиана SLA, мин",14154.00
7,Пропуски после заполнения,5040


In [58]:
# ПОДГОТОВКА ЕДИНОГО ФАЙЛА ДЛЯ ОБНОВЛЕНИЯ КОНТАКТОВ (01_contacts)
# Собираем всё в один DataFrame: 
# 1. Признак Buyer (был ли факт перехода: Оплата > 0 И Обучение > 0)
# 2. Дата перехода (согласно вашему запросу: момент ПЕРВОЙ такой успешной сделки)
# 3. Дата самой первой активности (даже лид-сделки) для исправления регистрации

BUYERS_INFO_PATH = os.path.join('..', 'data', 'cleaned', 'buyers_info.pkl')

# --- 1. Агрегация по всем сделкам ---
# Ищем самую первую активность (любая сделка, включая Lead/Lost)
reg_fixes_all = (
    df.groupby('contact_id')
    .agg(first_any_deal_date=('created_time', 'min'))
    .reset_index()
)

# --- 2. Агрегация по успешным сделкам (Buyer) ---
# ФАКТ ПЕРЕХОДА В КЛИЕНТА: Оплата была И сервис (обучение) предоставлен
df['is_buyer_deal'] = (df['initial_amount_paid'] > 0) & (df['months_of_study'] > 0)

# Ищем момент, когда лид СТАЛ клиентом (дата первой сделки, где Оплата > 0 и Сервис > 0)
buyers_only = (
    df[df['is_buyer_deal']]
    .groupby('contact_id')
    .agg(first_payment_date=('created_time', 'min'))
    .reset_index()
)
buyers_only['is_buyer'] = True

# --- 3. Слияние данных ---
contacts_update = reg_fixes_all.merge(buyers_only, on='contact_id', how='left')
contacts_update['is_buyer'] = contacts_update['is_buyer'].fillna(False)

# --- 4. Фикс аномалий регистрации ---
if os.path.exists(CONTACTS_CLEAN):
    contacts_reg = pd.read_pickle(CONTACTS_CLEAN)[['id', 'created_time']]
    contacts_update = contacts_update.merge(
        contacts_reg, 
        left_on='contact_id', 
        right_on='id', 
        how='left'
    ).drop(columns='id').rename(columns={'created_time': 'contact_created_time'})
    
    # Считаем аномалией, если любая сделка создана РАНЬШЕ записи о контакте в CRM
    contacts_update['new_registration_date'] = np.where(
        contacts_update['first_any_deal_date'] < contacts_update['contact_created_time'],
        contacts_update['first_any_deal_date'],
        pd.NaT
    )

# Сохраняем единый файл для синхронизации с блокнотом 01_contacts
cols_to_export = [
    'contact_id', 'is_buyer', 'first_payment_date', 
    'first_any_deal_date', 'new_registration_date'
]
contacts_update[cols_to_export].to_pickle(BUYERS_INFO_PATH)

n_buyers = contacts_update['is_buyer'].sum()
n_fixes = contacts_update['new_registration_date'].notna().sum()

print(f"Экспортирован файл: {BUYERS_INFO_PATH}")

print(f"Стали покупателями (Оплата + Обучение): {n_buyers}")
print(f"Подготовлено исправлений даты регистрации: {n_fixes}")

display(contacts_update[contacts_update['is_buyer']].head())

Экспортирован файл: ..\data\cleaned\buyers_info.pkl
Стали покупателями (Оплата + Обучение): 825
Подготовлено исправлений даты регистрации: 23


,contact_id,first_any_deal_date,first_payment_date,is_buyer,contact_created_time,new_registration_date
0,-1,2023-08-07 16:52:00,2023-11-20 13:12:00,True,NaT,NaT
3,5805028000000939010,2023-07-04 10:11:00,2023-07-04 10:11:00,True,2023-07-04 10:11:00,NaT
40,5805028000001350049,2023-07-08 08:56:00,2023-07-08 08:56:00,True,2023-07-08 08:55:00,NaT
68,5805028000001404153,2023-07-10 18:41:00,2024-01-17 19:27:00,True,2023-07-10 18:41:00,NaT
164,5805028000001880249,2023-07-15 13:27:00,2023-07-15 13:27:00,True,2023-07-15 13:27:00,NaT


## Описание датасета

**Источник:** `Deals (Done).xlsx` — выгрузка потенциальных сделок из CRM  
**Назначение:** основной датасет для анализа выручки, стадий продаж, воронки и LTV

### Ключевые аналитические показатели
| Столбец | Тип | Описание |
|---|---|---|
| `id` | `int64` | Уникальный ID сделки в CRM (19-значный) |
| `contact_id` | `int64` | ID связанного контакта (связь с `contacts.id`). Для анонимных сделок = `-1` |
| `deal_owner_name` | `category` | Менеджер, ответственный за ведение сделки |
| `stage` | `category` | Текущая стадия сделки. Статус payment done говорит о том что сделка оплачена. |
| `stage_group` | `category` | **Группировка стадий:** Won/Paid, Lost, Active Sales, Marketing/Lead |
| `initial_amount_paid` | `float64` | **Сумма, которую клиент заплатил первым платежом**  |
| `offer_total_amount` | `float64` | Полная контрактная стоимость продукта |
| `created_time` | `datetime` | Дата и время создания сделки |
| `closing_date` | `datetime` | Дата завершения сделки (факт профита или отказа) |
| `sla` | `int32` | **Время ответа:** SLA это время ответа продажника на заявку контакта |
| `is_bayer` | `boolen` | **Тип клиента:** Buyer (с оплатами и начавший обучение) / Lead (без оплат) |

### Обработка пропусков и специфика колонок
В процессе очистки была проведена работа по восполнению данных (**Backfill**) и нормализации.

- **Нормализация уровня языка (`level_of_deutsch`):**
    - Исходные данные содержали 215 вариантов написания (кириллица, латиница, текст).
    - Применены регулярные выражения и маппинг кириллицы (`а2` -> `A2`, `б1` -> `B1`).
    - Значения приведены к международному стандарту (A0–C2). Неразборчивые записи помечены как `Unclear`.

- **Восполнение (Backfill):** 
    - `source`, `campaign`, `city`, `level_of_deutsch`, `deal_owner_name`: данные "протянуты" между сделками одного и того же клиента (по `contact_id`). Если у клиента в одной сделке был указан уровень языка или город, а в другой — нет, мы восстановили эти данные.
    - `course_duration`: частично восполнено на основе выбранного продукта (`product`).

- **Оставлено "как есть" (NaN):**
    - `course_duration` (~83% пропусков) и `months_of_study` (~96% пропусков).
    - **Причина:** Данные поля заполняются в CRM преимущественно для успешных сделок (`Won/Paid`). Попытка заполнить их для лидов (через среднее или моду) приведет к серьезному искажению аналитики по продуктовой линейке и LTV. 
    - **closing_date** (~32% пропусков): Отсутствие даты означает, что сделка всё еще находится в работе (не закрыта ни в плюс, ни в минус).

- **Заполнение расчетными значениями:**
    - **months_of_study**: для всех сделок в статусе `Lost` проставлено `0`, так как обучения не было.
    - **closing_date**: для сделок в статусе `Lost` с отсутствующей датой закрытия проставлена дата создания (`created_time`), чтобы корректно учитывать их как завершенные в воронке.

- **Заполнено "Unknown":**
    - Категориальные поля (`quality`, `product`, `payment_type` и др.), где отсутствие информации является результатом отсутствия ввода данных в CRM.

**Ключевые связи:**
- `contact_id` → [01_cleaning_contacts.ipynb](01_cleaning_contacts.ipynb) (`id`)
- `stage_group` → фильтр для конверсии и ROMI


## 13. Описательная статистика

Анализ очищенных данных для проверки распределений и качества заполнения.

In [62]:
# 1. Сводная статистика для числовых полей
numeric_cols = ['initial_amount_paid', 'offer_total_amount', 'course_duration', 'months_of_study', 'sla']

def get_stats(data):
    stats = data[numeric_cols].describe().T
    stats['median'] = data[numeric_cols].median()
    stats['mode'] = data[numeric_cols].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    return stats[['mean', 'median', 'mode', 'min', 'max', 'std']].round(2)

print("--- ОБЩАЯ СТАТИСТИКА (все сделки) ---")
display(get_stats(df))

# Статистика только по успешным сделкам
won_df = df[df['stage_group'] == 'Won/Paid']
if not won_df.empty:
    print("\n--- СТАТИСТИКА УСПЕШНЫХ СДЕЛОК (Won/Paid) ---")
    display(get_stats(won_df))
else:
    print("\n[!] Успешные сделки не найдены.")

# 2. Анализ категориальных полей
cat_cols = ['quality', 'stage_group', 'source', 'product']

print("\n--- АНАЛИЗ КАТЕГОРИАЛЬНЫХ ПОЛЕЙ (Топ-10) ---")
for col in cat_cols:
    if col in df.columns:
        counts = df[col].value_counts(dropna=False)
        pct = (df[col].value_counts(normalize=True, dropna=False) * 100).round(1)
        
        stat_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pct}).head(10)
        print(f"\nПоле: {col}")
        display(stat_df)

--- ОБЩАЯ СТАТИСТИКА (все сделки) ---


,mean,median,mode,min,max,std
initial_amount_paid,196.13,0.00,0.00,0.00,11500.00,748.23
offer_total_amount,1616.39,0.00,0.00,0.00,11500.00,3720.91
course_duration,1.82,0.00,0.00,0.00,11.00,3.98
months_of_study,0.23,0.00,0.00,0.00,11.00,1.25
sla,21765.94,14154.00,11327.00,3.00,86292.00,20909.75



--- СТАТИСТИКА УСПЕШНЫХ СДЕЛОК (Won/Paid) ---


,mean,median,mode,min,max,std
initial_amount_paid,1141.53,1000.00,1000.00,0.00,11500.00,1477.72
offer_total_amount,7386.45,11000.00,11000.00,0.00,11500.00,3860.52
course_duration,9.97,11.00,11.00,0.00,11.00,2.34
months_of_study,5.33,5.00,6.00,0.00,11.00,2.99
sla,21999.49,13268.00,13268.00,32.00,85129.00,22405.60



--- АНАЛИЗ КАТЕГОРИАЛЬНЫХ ПОЛЕЙ (Топ-10) ---

Поле: quality


...

,Count,Percentage (%)
stage_group,,
Lost,13996,70.60
Active Sales,2808,14.20
Marketing/Lead,2155,10.90
Won/Paid,856,4.30



Поле: source


,Count,Percentage (%)
source,,
Facebook Ads,4728,23.90
Google Ads,4114,20.80
Tiktok Ads,2003,10.10
SMM,1668,8.40
Youtube Ads,1618,8.20
Organic,1498,7.60
CRM,1455,7.30
Bloggers,1074,5.40
Telegram posts,993,5.00



Поле: product


,Count,Percentage (%)
product,,
Unknown,16278,82.10
Digital Marketing,1953,9.90
UX/UI Design,1013,5.10
Web Developer,567,2.90
Find yourself in IT,3,0.00
Data Analytics,1,0.00


In [68]:
# ИССЛЕДОВАНИЕ АНОМАЛИИ: Успешные сделки с нулевым платежом
# Выводим сделки в статусе Won/Paid, где initial_amount_paid == 0
zero_paid_won = df[(df['stage_group'] == 'Won/Paid') & (df['initial_amount_paid'] == 0)]

print(f"Найдено успешных сделок с 0 платежом: {len(zero_paid_won)}")

if not zero_paid_won.empty:
    cols_to_show = ['id', 'contact_id', 'stage', 'product', 'initial_amount_paid', 'offer_total_amount', 'months_of_study']
    display(zero_paid_won[cols_to_show].head(20))
    # display(zero_paid_won.head(20))

    # Проверим распределение по стадиям внутри этой группы
    print("\nРаспределение по конкретным стадиям внутри 'Won/Paid' с 0 платежом:")
    display(zero_paid_won['stage'].value_counts().to_frame())
else:
    print("Аномалии не обнаружены.")

Найдено успешных сделок с 0 платежом: 18


,id,contact_id,stage,product,initial_amount_paid,offer_total_amount,months_of_study
15225,5805028000019345087,-1,Payment Done,Unknown,0.00,0.00,0
14007,5805028000022036007,-1,Payment Done,Unknown,0.00,0.00,0
21111,5805028000003130171,5805028000003112233,Payment Done,Digital Marketing,0.00,11000.00,11
20292,5805028000005180943,5805028000003660028,Payment Done,Unknown,0.00,0.00,0
19145,5805028000008675339,5805028000008684403,Payment Done,Unknown,0.00,0.00,0
18874,5805028000009352292,5805028000009349208,Payment Done,Unknown,0.00,0.00,0
18876,5805028000009352240,5805028000009363192,Payment Done,Unknown,0.00,0.00,0
18855,5805028000009349380,5805028000009371512,Payment Done,Unknown,0.00,0.00,0
18723,5805028000009670138,5805028000009566649,Payment Done,Unknown,0.00,0.00,0
18594,5805028000010015047,5805028000009995125,Payment Done,Unknown,0.00,0.00,0



Распределение по конкретным стадиям внутри 'Won/Paid' с 0 платежом:


,count
stage,
Payment Done,18
Call Delayed,0
Free Education,0
Lost,0
Need To Call,0
Need a consultation,0
Need to Call - Sales,0
New Lead,0
Qualificated,0
